# Pipeline ETL End-to-End (Medallion Architecture: Bronze → Silver → Gold)
**Mục tiêu:** Chuẩn hóa dữ liệu Myntra Fashion (43,982 bản ghi, 4 lớp `Casual`, `Ethnic`, `Formal`, `Sports`), đồng bộ tập chia `split_ids.csv`, kiểm tra chất lượng dữ liệu, lưu trữ dạng Delta Lake và xuất file `gold_metadata.parquet` cho nhánh Deep Learning trên Kaggle.

## 1. Cài đặt thư viện, Mount Drive & Khởi tạo SparkSession (Delta Lake)
* Cài đặt `pyspark` và `delta-spark`.
* Kết nối Google Drive và cấu hình tập trung toàn bộ đường dẫn Lakehouse (đảm bảo tính *idempotent* khi chạy lại nhiều lần).

In [1]:
!pip install -q pyspark delta-spark

import os, glob, shutil
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, lower, concat_ws, lit, coalesce, count, when, isnan, isnull
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from delta import configure_spark_with_delta_pip, DeltaTable

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Khởi tạo SparkSession tích hợp Delta Lake
builder = SparkSession.builder.appName("MultiModal_ETL_EndToEnd") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Spark Version: {spark.version}")

# 3. Cấu hình đường dẫn Lakehouse tập trung
LAKEHOUSE_DIR = '/content/drive/MyDrive/BigData_MultiModal_Project/lakehouse'
BRONZE_DIR    = f'{LAKEHOUSE_DIR}/bronze'
SILVER_DIR    = f'{LAKEHOUSE_DIR}/silver'
GOLD_DIR      = f'{LAKEHOUSE_DIR}/gold'

RAW_CSV_PATH        = f'{BRONZE_DIR}/styles.csv'
SPLIT_CSV_PATH      = f'{SILVER_DIR}/split_ids.csv'
GOLD_DELTA_PATH     = f'{GOLD_DIR}/fashion_multimodal_delta'
GOLD_PARQUET_TMP    = f'{GOLD_DIR}/gold_metadata_export'
GOLD_PARQUET_FINAL  = f'{GOLD_DIR}/gold_metadata.parquet'

os.makedirs(GOLD_DIR, exist_ok=True)

Mounted at /content/drive
Spark Version: 4.0.5


## 2. Xử lý tầng Bronze → Silver (Làm sạch, lọc 4 lớp & mã hóa StringIndexer)
* Đọc dữ liệu thô `styles.csv` từ tầng Bronze, loại bỏ NULL ở các cột cốt lõi (`id`, `usage`, `productDisplayName`).
* Chuẩn hóa văn bản (xóa thẻ HTML, chuyển chữ thường, xóa khoảng trắng thừa) và điền giá trị mặc định cho 7 cột Tabular.
* Lọc đúng 4 nhãn mục tiêu (`Casual`, `Sports`, `Ethnic`, `Formal`) và mã hóa các cột phân loại bằng `StringIndexer`.

In [2]:
# 1. Đọc dữ liệu thô từ tầng Bronze
raw_df = spark.read.csv(RAW_CSV_PATH, header=True, inferSchema=True)

# 2. Lọc bỏ NULL ở các cột cốt lõi & tạo tên file ảnh
clean_df = raw_df.dropna(subset=['id', 'usage', 'productDisplayName']) \
                 .withColumn("id_str", col("id").cast("string")) \
                 .withColumn("image_file", concat_ws(".", col("id_str"), lit("jpg"))) \
                 .withColumnRenamed("usage", "label")

# 3. Làm sạch chuỗi Text (xóa thẻ HTML, lowercase, gộp khoảng trắng thừa)
df_parsed = clean_df.withColumn("text", lower(regexp_replace(col("productDisplayName"), "<[^>]+>", " "))) \
                    .withColumn("text", regexp_replace(col("text"), "\\s+", " "))

# 4. Xử lý giá trị mặc định cho 7 cột Tabular
df_parsed = df_parsed.withColumn("gender", coalesce(col("gender"), lit("Unknown"))) \
                     .withColumn("masterCategory", coalesce(col("masterCategory"), lit("Unknown"))) \
                     .withColumn("subCategory", coalesce(col("subCategory"), lit("Unknown"))) \
                     .withColumn("articleType", coalesce(col("articleType"), lit("Unknown"))) \
                     .withColumn("baseColour", coalesce(col("baseColour"), lit("Unknown"))) \
                     .withColumn("season", coalesce(col("season"), lit("Unknown"))) \
                     .withColumn("year", coalesce(col("year").cast("double"), lit(2012.0)))

# 5. Lọc đúng 4 lớp Target Label đã chốt
top_classes = ['Casual', 'Sports', 'Ethnic', 'Formal']
df_filtered = df_parsed.filter(col("label").isin(top_classes))

# 6. Áp dụng StringIndexer cho các cột categorical & nhãn
categorical_cols = ['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'label']
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]

df_indexed = Pipeline(stages=indexers).fit(df_filtered).transform(df_filtered)
print(f"Hoàn tất làm sạch & mã hóa Silver. Số dòng trước khi join split: {df_indexed.count()}")

Hoàn tất làm sạch & mã hóa Silver. Số dòng trước khi join split: 44006


## 3. Đồng bộ `split_ids.csv` & Kiểm tra chất lượng dữ liệu (Data Quality Check)
* Thực hiện `inner join` với bảng `split_ids.csv` (đã đóng băng `seed=42`) để đảm bảo khớp 100% ID với các file đặc trưng `.pt`.
* Tự động kiểm thử 3 tiêu chí chất lượng: tổng số bản ghi (`43,982`), số ID trùng lặp (`0`), và giá trị `NULL`/`NaN` trên toàn bộ schema.

In [3]:
# 1. Join với split_ids.csv đã đóng băng ở Tuần 1
split_df = spark.read.csv(SPLIT_CSV_PATH, header=True) \
                     .select(col("id").cast("string").alias("split_id"), "split")

final_df = df_indexed.join(split_df, df_indexed.id_str == split_df.split_id, "inner").drop("split_id")

# 2. Báo cáo kiểm tra chất lượng dữ liệu (Data Quality Check)
print("--- BÁO CÁO KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY CHECK) ---")
total_count = final_df.count()
dup_count   = total_count - final_df.select("id").distinct().count()

print(f"1. Tổng số bản ghi: {total_count} (Kỳ vọng: 43,982)")
print(f"2. Số bản ghi trùng lặp ID: {dup_count} (Kỳ vọng: 0)")

numeric_types = ["double", "float"]
null_conditions = [
    count(when(isnull(f.name) | isnan(f.name) if f.dataType.simpleString() in numeric_types else isnull(f.name), f.name)).alias(f.name)
    for f in final_df.schema.fields
]
null_counts = final_df.select(null_conditions).collect()[0].asDict()

print("3. Kiểm tra giá trị NULL/NaN:")
has_null = False
for k, v in null_counts.items():
    if v > 0:
        print(f"   - Cột '{k}': {v} lỗi NULL")
        has_null = True
if not has_null:
    print("   -> Đạt chuẩn: Không có cột nào chứa NULL/NaN")

print("\n4. Phân bố theo tập split:")
final_df.groupBy("split").count().show()

# 3. Khóa kiểm thử tự động (Assertion)
assert total_count == 43982, f"Sai tổng số dòng! Kỳ vọng 43982, thực tế {total_count}"
assert dup_count == 0, f"Phát hiện {dup_count} ID bị trùng lặp!"
assert not has_null, "Phát hiện dữ liệu NULL/NaN trong bảng Gold!"
print("✅ PASSED: Dữ liệu Gold đạt chuẩn 100%!")

--- BÁO CÁO KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY CHECK) ---
1. Tổng số bản ghi: 43982 (Kỳ vọng: 43,982)
2. Số bản ghi trùng lặp ID: 0 (Kỳ vọng: 0)
3. Kiểm tra giá trị NULL/NaN:
   -> Đạt chuẩn: Không có cột nào chứa NULL/NaN

4. Phân bố theo tập split:
+-----+-----+
|split|count|
+-----+-----+
|train|30787|
| test| 6598|
|  val| 6597|
+-----+-----+

✅ PASSED: Dữ liệu Gold đạt chuẩn 100%!


## 4. Ghi tầng Gold (Delta Lake), Tối ưu Z-Order & Kiểm tra Lịch sử Versioning
* Ghi dữ liệu xuống tầng Gold ở định dạng Delta Lake (`.mode("overwrite")`, phân vùng theo cột `split` để tối ưu truy vấn Train/Val/Test).
* Chạy `OPTIMIZE ... ZORDER BY (label_idx)` để gom cụm dữ liệu vật lý và in bảng `DeltaTable.history()` làm minh chứng báo cáo.

In [4]:
# 1. Ghi đè xuống tầng Gold (Delta Lake, phân vùng theo split)
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("split") \
    .option("overwriteSchema", "true") \
    .save(GOLD_DELTA_PATH)

print(f"Đã lưu thành công Delta Lake tại: {GOLD_DELTA_PATH}")

# 2. Tối ưu hóa ổ đĩa bằng Z-Order theo nhãn (label_idx)
spark.sql(f"OPTIMIZE delta.`{GOLD_DELTA_PATH}` ZORDER BY (label_idx)")

# 3. In lịch sử phiên bản Delta Lake (ACID Versioning)
delta_table = DeltaTable.forPath(spark, GOLD_DELTA_PATH)
print("\n--- LỊCH SỬ THAO TÁC DELTA LAKE ---")
delta_table.history(5).select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

Đã lưu thành công Delta Lake tại: /content/drive/MyDrive/BigData_MultiModal_Project/lakehouse/gold/fashion_multimodal_delta

--- LỊCH SỬ THAO TÁC DELTA LAKE ---
+-------+-------------------+---------+----------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                         |
+-------+-------------------+---------+----------------------------------------------------------------------------+
|3      |2026-11-21 07:07:40|OPTIMIZE |{predicate -> [], zOrderBy -> ["label_idx"], clusterBy -> [], auto -> false}|
|2      |2026-11-21 07:07:12|WRITE    |{mode -> Overwrite, partitionBy -> ["split"], canOverwriteSchema -> true}   |
|1      |2026-11-18 16:04:52|OPTIMIZE |{predicate -> [], zOrderBy -> ["label"], clusterBy -> [], auto -> false}    |
|0      |2026-11-18 16:04:18|WRITE    |{mode -> Overwrite, partitionBy -> [], canOverwriteSchema -> true}          |
+-------+-----------

## 5. Xuất file tĩnh `gold_metadata.parquet` cho nhánh Deep Learning (Kaggle)
* Gom toàn bộ bảng Gold về 1 file Parquet duy nhất (`coalesce(1)`) và lưu tại đường dẫn cố định để đồng bộ sang Kaggle GPU.

In [5]:
# Xuất ra 1 file Parquet duy nhất (Idempotent nhờ mode overwrite)
final_df.coalesce(1).write.mode("overwrite").parquet(GOLD_PARQUET_TMP)

part_file = glob.glob(f"{GOLD_PARQUET_TMP}/part-*.parquet")[0]
shutil.copy(part_file, GOLD_PARQUET_FINAL)

print(f"✅ Đã xuất xong file Parquet tĩnh tại: {GOLD_PARQUET_FINAL}")

✅ Đã xuất xong file Parquet tĩnh tại: /content/drive/MyDrive/BigData_MultiModal_Project/lakehouse/gold/gold_metadata.parquet
